# 04 — Gold: LAD Summary Aggregation
Aggregates enriched listings up to one row per LAD — the final summary table used by the investment scoring engine and Streamlit app.

**Catalog:** `airbnb_app`  
**Reads from:** `airbnb_app.gold.airbnb_listings_{city}` (output of `03_gold_lad.ipynb`), `airbnb_app.clean.rent_data`  
**Writes to:** `airbnb_app.gold.lad_summary`  

| Metric | Definition |
|--------|------------|
| `listing_count` | Number of active Airbnb listings in the LAD |
| `median_house_price` | Median MSOA house price, aggregated to LAD (from `03_gold_lad.ipynb`) |
| `median_annual_rent` | Median yearly long-term rent, matched by bedroom count per listing then aggregated |
| `median_airbnb_income_ons` | Median `estimated_revenue_l365d` — Inside Airbnb's own pre-calculated estimate |
| `median_airbnb_income_calendar` | Median income calculated from calendar occupancy proxy × nightly price |

> Two Airbnb income estimates are shown side by side so any divergence between Inside Airbnb's own figure and the calendar-derived figure is visible and explainable.

## 0. Config

In [0]:
GOLD_DB  = "airbnb_app.gold"
CLEAN_DB = "airbnb_app.clean"

CITIES = ["london", "manchester", "edinburgh", "bristol"]

# Bedroom count to rent column mapping
# Listings with 4+ bedrooms use the four_or_more_bed rent figure
BEDROOM_RENT_MAP = {
    0: "rental_price_one_bed",     # studio treated as one bed for rent comparison
    1: "rental_price_one_bed",
    2: "rental_price_two_bed",
    3: "rental_price_three_bed",
}
# 4+ bedrooms and any null/unmatched bedroom count fall back to this column
RENT_FALLBACK_COL = "rental_price_four_or_more_bed"
RENT_OVERALL_COL  = "rental_price"  # used if bedroom-specific rent is null

# Most recent rent period to use (rent_data already filtered to last 12 months
# in the cleaning notebook — take the median across those months per area)

## 1. Setup

In [0]:
import pandas as pd
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_DB}")
print(f"Schema ready: {GOLD_DB}")

## 2. Load and union enriched listings across all cities

In [0]:
listings_dfs = []
for city in CITIES:
    df = spark.table(f"{GOLD_DB}.airbnb_listings_{city}")
    listings_dfs.append(df)

# Union all cities into one DataFrame
listings_all = listings_dfs[0]
for df in listings_dfs[1:]:
    listings_all = listings_all.unionByName(df, allowMissingColumns=True)

print(f"Total listings across all cities: {listings_all.count():,}")
listings_all.select("_city", "lad_code", "lad_name", "bedrooms", "price").show(5)

## 3. Calculate Airbnb income — calendar-derived occupancy proxy

Built from the calendar table: occupancy proxy = unavailable nights / 365, then estimated revenue = occupancy proxy × nightly price × 365.

> **Limitation:** `available = false` cannot distinguish a booked night from a host-blocked night. This is a proxy, not a confirmed booking count — stated clearly in the app's assumptions section.

In [0]:
occupancy_dfs = []

for city in CITIES:
    cal = spark.table(f"{CLEAN_DB}.airbnb_calendar_{city}")

    occ = (
        cal.groupBy("listing_id")
        .agg(
            F.count("*").alias("total_days"),
            F.sum(F.when(F.col("available") == False, 1).otherwise(0)).alias("unavailable_days"),
        )
        .withColumn("occupancy_proxy", F.col("unavailable_days") / F.col("total_days"))
        .withColumn("_city", F.lit(city))
    )
    occupancy_dfs.append(occ)

occupancy_all = occupancy_dfs[0]
for df in occupancy_dfs[1:]:
    occupancy_all = occupancy_all.unionByName(df, allowMissingColumns=True)

print(f"Occupancy proxy calculated for {occupancy_all.count():,} listings")
occupancy_all.show(5)

In [0]:
# Join occupancy proxy to listings and calculate calendar-derived income estimate
listings_all = listings_all.join(
    occupancy_all.select("listing_id", "occupancy_proxy"),
    listings_all["id"] == occupancy_all["listing_id"],
    how="left"
).drop("listing_id")

listings_all = listings_all.withColumn(
    "airbnb_income_calendar",
    F.col("occupancy_proxy") * F.col("price") * 365
)

print("Calendar-derived income estimate added.")
listings_all.select("id", "price", "occupancy_proxy", "airbnb_income_calendar", "estimated_revenue_l365d").show(5)

## 4. Match rent by bedroom count

Long-term rent data has separate columns for 1/2/3/4+ bedroom properties. Each listing is matched to the comparable rent figure based on its own `bedrooms` value, then this gets aggregated up to LAD level for a fair like-for-like comparison.

In [0]:
# Load rent data — already filtered to most recent 12 months in cleaning notebook
# Take median across those months per area for a stable annual figure
rent_raw = spark.table(f"{CLEAN_DB}.rent_data")

rent_by_area = (
    rent_raw.groupBy("area_code", "area_name")
    .agg(
        F.expr("percentile_approx(rental_price, 0.5)").alias("rent_overall"),
        F.expr("percentile_approx(rental_price_one_bed, 0.5)").alias("rent_one_bed"),
        F.expr("percentile_approx(rental_price_two_bed, 0.5)").alias("rent_two_bed"),
        F.expr("percentile_approx(rental_price_three_bed, 0.5)").alias("rent_three_bed"),
        F.expr("percentile_approx(rental_price_four_or_more_bed, 0.5)").alias("rent_four_plus_bed"),
    )
)

print(f"Rent data aggregated to {rent_by_area.count():,} areas (median of last 12 months)")
rent_by_area.show(5)

In [0]:
# Join rent data to listings on lad_code = area_code
listings_all = listings_all.join(
    rent_by_area,
    listings_all["lad_code"] == rent_by_area["area_code"],
    how="left"
)

# For each listing, pick the rent figure matching its bedroom count
# 0-1 bed -> one_bed, 2 bed -> two_bed, 3 bed -> three_bed, 4+ -> four_plus_bed
# Fall back to overall rent if the bedroom-specific figure is null
listings_all = listings_all.withColumn(
    "matched_monthly_rent",
    F.when(F.col("bedrooms").isNull(), F.col("rent_overall"))
     .when(F.col("bedrooms") <= 1, F.coalesce(F.col("rent_one_bed"), F.col("rent_overall")))
     .when(F.col("bedrooms") == 2, F.coalesce(F.col("rent_two_bed"), F.col("rent_overall")))
     .when(F.col("bedrooms") == 3, F.coalesce(F.col("rent_three_bed"), F.col("rent_overall")))
     .otherwise(F.coalesce(F.col("rent_four_plus_bed"), F.col("rent_overall")))
)

# Annualise — rent data is monthly
listings_all = listings_all.withColumn(
    "matched_annual_rent",
    F.col("matched_monthly_rent") * 12
)

print("Bedroom-matched rent added.")
listings_all.select("id", "bedrooms", "matched_monthly_rent", "matched_annual_rent").show(10)

In [0]:
# ── Section 4b: Rail station accessibility ──────────────────────────────────
# Rail station walk-time data is at MSOA level — aggregate to LAD by median,
# same pattern as house prices

rail_pd = spark.table(f"{CLEAN_DB}.amenities_rail_stations").toPandas()

# Need MSOA → LAD mapping to aggregate up
# Reuse the house prices table which already has both codes
msoa_to_lad = (
    spark.table(f"{CLEAN_DB}.house_prices_msoa")
    .select("msoa_code", "local_authority_code")
    .distinct()
    .toPandas()
    .rename(columns={"local_authority_code": "lad_code"})
)

rail_with_lad = rail_pd.merge(msoa_to_lad, on="msoa_code", how="left")

rail_by_lad = (
    rail_with_lad
    .groupby("lad_code", as_index=False)
    .agg(
        pct_within_15min_rail=("less_than_15_minute_walk", "median"),
        pct_within_30min_rail=("less_than_30_minute_walk", "median"),
        pct_within_60min_rail=("less_than_60_minute_walk", "median"),
    )
)

print(f"Rail accessibility aggregated to {len(rail_by_lad):,} LADs")
rail_by_lad.head()

## 5. Aggregate to LAD summary

In [0]:
LAD_SUMMARY_TABLE = f"{GOLD_DB}.lad_summary"

lad_summary = (
    listings_all
    .filter(F.col("lad_code").isNotNull())
    .groupBy("lad_code", "lad_name", "_city")
    .agg(
        F.count("*").alias("listing_count"),

        # House prices — already at LAD level from 03_gold_lad.ipynb
        F.expr("percentile_approx(median_house_price_2025, 0.5)").alias("median_house_price"),
        F.expr("percentile_approx(price_growth_10yr, 0.5)").alias("median_price_growth_10yr"),

        # Rent — bedroom-matched, annualised
        F.expr("percentile_approx(matched_annual_rent, 0.5)").alias("median_annual_rent"),

        # Airbnb income — two estimates side by side
        F.expr("percentile_approx(estimated_revenue_l365d, 0.5)").alias("median_airbnb_income_ons"),
        F.expr("percentile_approx(airbnb_income_calendar, 0.5)").alias("median_airbnb_income_calendar"),

        # Supporting context metrics
        F.expr("percentile_approx(price, 0.5)").alias("median_nightly_price"),
        F.expr("percentile_approx(review_scores_rating, 0.5)").alias("median_review_score"),
        F.expr("percentile_approx(occupancy_proxy, 0.5)").alias("median_occupancy_proxy"),
        F.expr("percentile_approx(gp_surgery_count, 0.5)").alias("gp_surgery_count"),
        F.expr("percentile_approx(total_parks_count, 0.5)").alias("total_parks_count"),
        F.expr("percentile_approx(pct_within_15min_rail, 0.5)").alias("pct_within_15min_rail"),
        F.expr("percentile_approx(pct_within_30min_rail, 0.5)").alias("pct_within_30min_rail"),
    )
)

# Calculate STR vs LTR yield using the median house price as the investment base
lad_summary = lad_summary.withColumn(
    "str_gross_yield",
    F.when(
        F.col("median_house_price").isNotNull() & (F.col("median_house_price") > 0),
        F.col("median_airbnb_income_calendar") / F.col("median_house_price")
    )
).withColumn(
    "ltr_gross_yield",
    F.when(
        F.col("median_house_price").isNotNull() & (F.col("median_house_price") > 0),
        F.col("median_annual_rent") / F.col("median_house_price")
    )
).withColumn(
    "yield_gap",
    F.col("str_gross_yield") - F.col("ltr_gross_yield")
)

lad_summary = lad_summary.withColumn("_gold_created_at", F.current_timestamp())

(
    lad_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(LAD_SUMMARY_TABLE)
)

row_count = lad_summary.count()
print(f"✓ {LAD_SUMMARY_TABLE} ({row_count:,} LADs)")

## 6. Spot checks

In [0]:
# Full summary table sorted by listing count
spark.sql("""
    SELECT lad_name, _city, listing_count,
           ROUND(median_house_price, 0)            AS median_house_price,
           ROUND(median_annual_rent, 0)             AS median_annual_rent,
           ROUND(median_airbnb_income_ons, 0)       AS airbnb_income_ons,
           ROUND(median_airbnb_income_calendar, 0)  AS airbnb_income_calendar,
           ROUND(str_gross_yield * 100, 2)          AS str_yield_pct,
           ROUND(ltr_gross_yield * 100, 2)          AS ltr_yield_pct,
           ROUND(yield_gap * 100, 2)                AS yield_gap_pct
    FROM airbnb_app.gold.lad_summary
    ORDER BY listing_count DESC
    LIMIT 20
""").display()

In [0]:
# Compare the two Airbnb income estimates — flag large divergences
spark.sql("""
    SELECT lad_name, _city,
           ROUND(median_airbnb_income_ons, 0)      AS income_ons,
           ROUND(median_airbnb_income_calendar, 0) AS income_calendar,
           ROUND(
               ABS(median_airbnb_income_ons - median_airbnb_income_calendar)
               / median_airbnb_income_ons * 100, 1
           ) AS pct_divergence
    FROM airbnb_app.gold.lad_summary
    WHERE median_airbnb_income_ons IS NOT NULL
      AND median_airbnb_income_calendar IS NOT NULL
    ORDER BY pct_divergence DESC
    LIMIT 15
""").display()

In [0]:
# Check Edinburgh — should have listing_count and median_airbnb_income
# but null house price / rent (England & Wales only datasets)
spark.sql("""
    SELECT lad_name, listing_count,
           median_house_price, median_annual_rent,
           median_airbnb_income_calendar
    FROM airbnb_app.gold.lad_summary
    WHERE _city = 'edinburgh'
    ORDER BY listing_count DESC
""").display()

In [0]:
# Export to CSV for Streamlit app consumption
EXPORT_PATH = "/Volumes/airbnb_app/gold/exports/lad_summary.csv"

import os
os.makedirs(os.path.dirname(EXPORT_PATH), exist_ok=True)

lad_summary_pd = spark.table("airbnb_app.gold.lad_summary").toPandas()
lad_summary_pd.to_csv(EXPORT_PATH, index=False)

print(f"✓ Exported {len(lad_summary_pd):,} rows to {EXPORT_PATH}")

In [0]:
# Also export to /tmp for download via Databricks UI
# Click the file in the left sidebar's Data tab, or use dbutils.fs.cp to move it
# to a location you can download from

LOCAL_DOWNLOAD_PATH = "/tmp/lad_summary.csv"
lad_summary_pd.to_csv(LOCAL_DOWNLOAD_PATH, index=False)

print(f"✓ Saved to {LOCAL_DOWNLOAD_PATH}")
print("To download: use the Databricks file browser, or run the cell below to display a direct download link.")

In [0]:
# Generate a downloadable link directly in the notebook
import base64

with open(LOCAL_DOWNLOAD_PATH, "rb") as f:
    csv_data = f.read()

b64 = base64.b64encode(csv_data).decode()
download_link = f'<a href="data:text/csv;base64,{b64}" download="lad_summary.csv">Click here to download lad_summary.csv</a>'

displayHTML(download_link)

## Notes

**Two Airbnb income estimates, shown side by side:**
- `median_airbnb_income_ons` — Inside Airbnb's own pre-calculated `estimated_revenue_l365d`, treated as an external benchmark.
- `median_airbnb_income_calendar` — built independently from the calendar table: occupancy proxy (unavailable nights ÷ 365) × nightly price × 365.
- Where the two diverge significantly, investigate before trusting either figure blindly — both rely on assumptions about unavailable nights that may not reflect actual bookings.

**Rent matched by bedroom count:**
- Each listing is matched to the LAD's published rent for its own bedroom count (1/2/3/4+ bed). Studios (0 bed) are matched to the 1-bed rent figure as the closest comparable.
- If a bedroom-specific rent figure is missing for an LAD, the overall `rental_price` is used as a fallback.
- Rent data is monthly — annualised by × 12 for direct comparison with Airbnb annual income.

**Edinburgh data gaps:**
- `median_house_price` and `median_annual_rent` will be null for Edinburgh — house price and rent datasets cover England and Wales only.
- `listing_count` and both Airbnb income estimates will still populate correctly since these come from Inside Airbnb data which does cover Edinburgh.
- `str_gross_yield`, `ltr_gross_yield`, and `yield_gap` will be null for Edinburgh as a result — flag this clearly in the app UI.

**Yield calculation:**
- Gross yield = annual income ÷ purchase price. This is a simplified gross figure — no account for mortgage costs, management fees, void periods, or the London 90-night cap.
- `yield_gap` = STR yield − LTR yield. A positive value suggests Airbnb letting outperforms long-term renting on a gross basis, before regulatory and operational costs are considered.

**Next step:** This table (`airbnb_app.gold.lad_summary`) feeds directly into the investment scoring notebook and the Streamlit app's LAD comparison view.